In [12]:
from pprint import pprint as pp
from Bio import Entrez, SeqIO
from Bio.SeqRecord import SeqRecord, Seq

from pathlib import Path
from multiprocessing import Pool
from collections import defaultdict
import pandas as pd

from itertools import product


from gb2seq.alignment import Gb2Alignment, ReferenceInsertionError
from gb2seq.features import Features
from dark.fasta import FastaReads

In [10]:
!ls -l tests/resources/

total 408
-rw-rw-r-- 1 1005 1005  15483 Sep 16 12:23 consensus.fasta
-rw-r--r-- 1 1005 1005  32702 Sep 16 13:41 noro_gb_records.gb
-rw-r--r-- 1 1005 1005 357590 Sep 16 13:51 similarities.csv
drwxr-xr-x 2 1005 1005   4096 Sep 16 16:16 split_fastas
drwxr-xr-x 2 1005 1005   4096 Sep 16 16:16 split_genbank


In [6]:
df_sim = pd.read_csv('results/similarity_outputs/similarities.csv')

In [11]:
fasta_path = 'tests/resources/consensus.fasta'
genbank_path = 'tests/resources/noro_gb_records.gb'

with open(fasta_path) as fh:
    fasta_id_list = [record.id for record in SeqIO.parse(fh,'fasta')]
with open(genbank_path) as fh:
    gb_id_list = [record.id for record in SeqIO.parse(fh,'genbank')]

In [13]:
fasta_id_list, gb_id_list

(['Noro-P8-30-new-184', 'Noro-P3-30-new-182'], ['PZ526968.1', 'PQ207934.1'])

In [14]:
list(product(fasta_id_list, gb_id_list))

[('Noro-P8-30-new-184', 'PZ526968.1'),
 ('Noro-P8-30-new-184', 'PQ207934.1'),
 ('Noro-P3-30-new-182', 'PZ526968.1'),
 ('Noro-P3-30-new-182', 'PQ207934.1')]

In [20]:
df_sim.set_index(['consensus_id','genbank_id']).loc[list(product(fasta_id_list, gb_id_list))].sort_index()

identity
consensus_id       genbank_id          
Noro-P3-30-new-182 PQ207934.1  0.964531
                   PZ526968.1  0.974586
Noro-P8-30-new-184 PQ207934.1  0.959767
                   PZ526968.1  0.956861

In [37]:
df_sim.set_index('consensus_id').loc[l].reset_index().to_csv('tests/resources/similarities.csv', index=False)

In [40]:
!du -h tests/resources/

404K	tests/resources/


In [24]:
df_sim = df_sim.loc[
            df_sim.groupby("consensus_id")["identity"].idxmax(),
            ["consensus_id", "genbank_id"],
        ].set_index('consensus_id')
df_sim.head()

,genbank_id
consensus_id,
Noro-P1-20-180,PQ373147.1
Noro-P1-30-new-182,PQ373147.1
Noro-P10-30-new-184,MK764013.1
Noro-P100-30-new-186,PQ207934.1
Noro-P101-30-new-186,PQ207934.1


In [5]:
with open('tests/resources/consensus.fasta') as fh:
    l = [record.id for record in SeqIO.parse(fh, 'fasta')]

In [33]:
df_sim.loc[l]

,genbank_id
consensus_id,
Noro-P8-30-new-184,PQ207934.1
Noro-P3-30-new-182,PZ526968.1


In [ ]:
df_sim

In [4]:
with open('tests/resources/noro_gb_records.gb','rt') as f:
    for record in SeqIO.parse(f, 'genbank'):
        print(record.id)

PZ526968.1
PQ207934.1


In [31]:
desired_gb_ids = set(df_sim.loc[l].genbank_id.values)
desired_gb_ids

{'PQ207934.1', 'PZ526968.1'}

In [32]:
with open('tests/noro_gb_records.gb','wt') as fo:
    with open('reduced_references.gb', 'r') as f:
        for record in SeqIO.parse(f, 'genbank'):
            if record.id in desired_gb_ids:
                SeqIO.write(record, fo, 'genbank')

In [2]:
orf2_names = {'ORF-2',
 'ORF2',
 'VP1',
 'VP1 capsid protein',
 'VP1 protein',
 'capid protein VP1',
 'capsid VP1',
 'capsid protein VP1',
 'major capsid protein',
 'major capsid protein VP1',
 'major capsid region',
 'major structural protein',
 'major viral capsid protein',
 }
orf1_names = {
 'ORF-1',
 'ORF1',
 'ORF1 polyprotein',
 'non structural polyprotein',
 'non structural protein',
 'non-structrual polyprotein',
 'non-structural polyporotein precursor',
 'non-structural polyprotein',
 'non-structural polyprotein precursor',
 'non-structural protein',
 'non-structure protein',
 'non-strustural polyprotein',
 'nonstructrual polyprotein',
 'nonstructural polyprotein',
 'nonstructural protein'}

In [3]:
s1 = {
 'ORF-1',
 'ORF1',
 'ORF1 polyprotein',
 'non structural polyprotein',
 'non structural protein',
 'non-structrual polyprotein',
 'non-structural polyporote'}

In [4]:
def split_gb_into_files(fn, directory, orf1_names, orf2_names):
    out_dir = Path(directory)
    wrote_records, total_records = 0,0
    for i, record in enumerate(SeqIO.parse(fn, "genbank")):
        total_records += 1
        feature_names = set(product for feature in record.features if 'product' in feature.qualifiers for product in feature.qualifiers['product'])
        # gb2seq uses the "product" as feature name, so check beforehand if the gb entry does supply a valid one
        if (len(feature_names.intersection(orf1_names)) > 0) and (len(feature_names.intersection(orf2_names)) > 0):
            SeqIO.write(record, out_dir / f"{record.id}.gb", "genbank")
            wrote_records+=1
    print(f'wrote {wrote_records} out of total {total_records}')


def write_consensus_to_separate_files(fns, directory):
    out_dir = Path(directory)
    for fn in fns:
        for record in SeqIO.parse(fn, "fasta"):
            SeqIO.write(record, out_dir / f"{record.id}.fasta", "fasta")


def process_genbank(fn_list):
    fn_gb, fn_fasta = fn_list
    feat = Features(fn_gb)
    reads = list(FastaReads(fn_fasta))
    assert len(reads) == 1, f'fasta {fn_fasta} contains more than one fasta entry'
    read = reads[0]
    aln = Gb2Alignment(read, feat, aligner="edlib")

    ref_seq = aln.referenceAligned.sequence
    genome_seq = aln.genomeAligned.sequence
    identity = sum(
        a == b for a, b in zip(ref_seq, genome_seq)
    ) / len(ref_seq)

    return [feat.reference.id, read.id, identity]


In [21]:
fr = list(FastaReads('consensi_dir/Noro-P1-20-180.fasta'))[0]

In [5]:
email = "udo.gieraths@gmail.com"
api_key = "a6aa5a83e82d576d1959657448b3bf9ae408"

In [6]:
Entrez.email = email
Entrez.api_key = api_key

In [7]:
g2_4_ref_gb = "PP549876.1.gb"
g2_17_ref_gb =  "NC_039475.1.gb"

In [8]:
%%time
split_gb_into_files("noroII.gb", 'genbank_files', orf1_names=orf1_names, orf2_names=orf2_names)

wrote 3544 out of total 4373
CPU times: user 2.58 s, sys: 283 ms, total: 2.86 s
Wall time: 2.86 s


In [9]:
%%time
write_consensus_to_separate_files(['g2_4_consensus.fasta', 'g2_17_consensus.fasta'], 'consensi_dir')

CPU times: user 7.21 ms, sys: 7.04 ms, total: 14.3 ms
Wall time: 12.6 ms


In [10]:
%%time
all_gbs = list(Path("genbank_files").glob("*.gb"))
all_consensi = list(Path("consensi_dir").glob("*.fasta"))

all_combinations = [[fn_gb, fn_fasta] for fn_fasta in all_consensi for fn_gb in all_gbs]

maxd = {}
no_processors = 64

l = []
with Pool(no_processors) as pool:
    no_jobs = len(all_combinations)
    jobs_processed = 0
    
    # distribute amont the cores, such that 4 chunks are processed per processor
    chunksize = max(1, no_jobs // (no_processors * 4))
    
    # files are processed in parallel and the iterator 
    # yields results as soon as a job is done. The order is not preserved.
    for result in pool.imap_unordered(
        process_genbank, all_combinations, chunksize=chunksize):
        jobs_processed += 1

        l.append(result)
        ref, cons, sim = result
        if cons not in maxd or sim > maxd[cons][1]:
            maxd[cons] = [ref, sim]

        if jobs_processed % 10000 == 0:
            print(jobs_processed, f"{jobs_processed / len(all_combinations):.2%}")

10000 3.27%
20000 6.53%
30000 9.80%
40000 13.07%
50000 16.33%
60000 19.60%
70000 22.87%
80000 26.13%
90000 29.40%
100000 32.67%
110000 35.93%
120000 39.20%
130000 42.47%
140000 45.74%
150000 49.00%
160000 52.27%
170000 55.54%
180000 58.80%
190000 62.07%
200000 65.34%
210000 68.60%
220000 71.87%
230000 75.14%
240000 78.40%
250000 81.67%
260000 84.94%
270000 88.20%
280000 91.47%
290000 94.74%
300000 98.00%
CPU times: user 951 ms, sys: 458 ms, total: 1.41 s
Wall time: 35.6 s


In [12]:
l[:3]

[['MH260486.1',
  'Noro-P47-30-new-184 NC_039475.1_artic-network/amplicon-nf_0.8.0',
  0.4916374951380786],
 ['NC_039477.1',
  'Noro-P47-30-new-184 NC_039475.1_artic-network/amplicon-nf_0.8.0',
  0.5028497409326425],
 ['PV806172.1',
  'Noro-P47-30-new-184 NC_039475.1_artic-network/amplicon-nf_0.8.0',
  0.47127029116207164]]

In [19]:

def read_consensus_ids(fasta: Path) -> set[str]:
	"""Read all consensus IDs from FASTA input files."""
	consensus_ids: set[str] = set()

	count = 0
	for record in SeqIO.parse(fasta, "fasta"):
		consensus_ids.add(record.id)
		count += 1
	if count == 0:
		raise ValueError(f"FASTA file contains no records: {fasta}")

	if not consensus_ids:
		raise ValueError("No consensus IDs found in FASTA inputs.")

	return consensus_ids

def load_similarity_table(path: Path) -> pd.DataFrame:
	"""Load and normalize similarity CSV columns to ref/cons/sim."""
	df = cast(pd.DataFrame, pd.read_csv(path))
	columns = set(df.columns)

	if {"ref", "cons", "sim"}.issubset(columns):
		normalized = cast(pd.DataFrame, df[["ref", "cons", "sim"]].copy())
	elif {"genbank_path", "consensus_id", "identity"}.issubset(columns):
		normalized = cast(
			pd.DataFrame,
			df[["genbank_path", "consensus_id", "identity"]].copy(),
		)
		normalized.columns = ["ref", "cons", "sim"]
	else:
		raise ValueError(
			"Unsupported CSV columns. Expected either ref/cons/sim or "
			"genbank_path/consensus_id/identity."
		)

	normalized["ref"] = normalized["ref"].astype(str)
	normalized["cons"] = normalized["cons"].astype(str)
	normalized["sim"] = pd.to_numeric(normalized["sim"], errors="raise")
	return normalized
    

NameError: name 'pd' is not defined

In [25]:
a = open('similarities.csv').readlines()
a[:2]

['consensus_id,genbank_path,identity\n',
 'Noro-P9-30-new-184,/tmp/compute_similarities_02tce4qj/genbank_files/PZ679177.1.gb,0.33421435\n']

In [18]:
cis = read_consensus_ids('both_gt_consensus.fasta')
len(cis)

70

In [14]:
dd = defaultdict(list)
for ref, cons, sim in l[:]:
    dd[cons].append([ref,sim])
    
for topX in [3,5,7,10]:
    print(topX, len(set(item[0] for v in dd.values() for item in sorted(v, key=lambda x: -x[1])[:topX])))
    

3 50
5 76
7 98
10 130


In [54]:
all_consensi[:10]

[PosixPath('consensi_dir/Noro-P72-30-new-186.fasta'),
 PosixPath('consensi_dir/Noro-P12-30-new-184.fasta'),
 PosixPath('consensi_dir/Noro-P47-30-new-184.fasta'),
 PosixPath('consensi_dir/Noro-P45-30-new-184.fasta'),
 PosixPath('consensi_dir/Noro-P82-30-new-186.fasta'),
 PosixPath('consensi_dir/Noro-P111-30-new-186.fasta'),
 PosixPath('consensi_dir/Noro-P80-30-new-186.fasta'),
 PosixPath('consensi_dir/Noro-P74-30-new-186.fasta'),
 PosixPath('consensi_dir/Noro-P104-30-new-186.fasta'),
 PosixPath('consensi_dir/Noro-P2-30-new-187.fasta')]

In [52]:
sorted(dd[cons], key=lambda x: -x[1])

[['PV750882.1', 0.8297168563111934],
 ['PV750894.1', 0.8295845461762371],
 ['PV784324.1', 0.8293393353634317],
 ['PQ310522.1', 0.8293199259063244],
 ['PV750891.1', 0.8293199259063244],
 ['PV394769.1', 0.8292521508934481],
 ['PV750898.1', 0.8291876157713681],
 ['PV784303.1', 0.8291876157713681],
 ['PV784304.1', 0.8291876157713681],
 ['PV750886.1', 0.8290553056364117],
 ['PV784323.1', 0.8289229955014554],
 ['PV750885.1', 0.8289229955014554],
 ['PQ310130.1', 0.8288097444723951],
 ['PV750893.1', 0.828790685366499],
 ['PQ310518.1', 0.828790685366499],
 ['PV784347.1', 0.828790685366499],
 ['PZ234778.1', 0.8286773467496359],
 ['PV784344.1', 0.8286583752315427],
 ['PQ373143.1', 0.8285449490268767],
 ['PV750889.1', 0.8285260650965864],
 ['PV750881.1', 0.8285260650965864],
 ['PV750884.1', 0.8285260650965864],
 ['PQ310519.1', 0.8284164572033338],
 ['PQ310131.1', 0.8284125513041176],
 ['PQ373146.1', 0.8282801535813584],
 ['PV750883.1', 0.8282614448266737],
 ['PV784331.1', 0.8282614448266737],
 ['P

In [26]:
def yield_orf_sequences(fasta_file, feat_name):
    for read in FastaReads(fasta_file):
        clostes_seq = maxd[read.id][0]
        feat = Features(f"genbank_files/{clostes_seq}.gb")
        is_orf1 = feat_name in orf1_names
        is_orf2= feat_name in orf2_names
        assert is_orf1 or is_orf2, f'{feat_name} is neither associated with orf1 or orf2'
        # find any of the matching names for this feature, could also list a priority in case multiple match
        inter = orf1_names.intersection(feat.keys()) if is_orf1 else orf2_names.intersection(feat.keys())
        # for now, let's just pick one of the matching names, they should all represent the same thing
        feat_name = inter.pop()
        alignment = Gb2Alignment(read, feat)
        try:
            try:
                _, cut_orf = alignment.ntSequences(feat_name)
            except ReferenceInsertionError:
                print(f'ReferenceInsertion in {clostes_seq} for sequence {read.id} needed')
                _, cut_orf = alignment.ntSequences(feat_name, raiseOnReferenceGaps=False)
        except:
            print(clostes_seq, read.id)
            raise e
        yield SeqRecord(seq=Seq(cut_orf.sequence),
                        id=cut_orf.id.split(' ')[0],
                        description=cut_orf.id)            
        

In [27]:
%%time
with open('g2_4_consensus_orf1_gb2seqed.fasta', 'wt') as fh:
    SeqIO.write(yield_orf_sequences('g2_4_consensus.fasta', "ORF1"), fh, 'fasta')
    

CPU times: user 91 ms, sys: 26.1 ms, total: 117 ms
Wall time: 4.43 s


In [28]:
%%time
with open('g2_4_consensus_orf2_gb2seqed.fasta', 'wt') as fh:
    SeqIO.write(yield_orf_sequences('g2_4_consensus.fasta', "ORF2"), fh, 'fasta')

CPU times: user 83 ms, sys: 33.9 ms, total: 117 ms
Wall time: 4.43 s


In [30]:
%%time
with open('g2_17_consensus_orf2_gb2seqed.fasta', 'wt') as fh:
    SeqIO.write(yield_orf_sequences('g2_17_consensus.fasta', "ORF2"), fh, 'fasta')

ReferenceInsertion in PV750882.1 for sequence Noro-P111-30-new-186 NC_039475.1_artic-network/amplicon-nf_0.8.0 needed
ReferenceInsertion in PV750882.1 for sequence Noro-P19-30-new-184 NC_039475.1_artic-network/amplicon-nf_0.8.0 needed
ReferenceInsertion in PV750882.1 for sequence Noro-P14-30-new-184 NC_039475.1_artic-network/amplicon-nf_0.8.0 needed
ReferenceInsertion in PV750882.1 for sequence Noro-P34-30-new-184 NC_039475.1_artic-network/amplicon-nf_0.8.0 needed
ReferenceInsertion in PQ373143.1 for sequence Noro-P88-30-new-186 NC_039475.1_artic-network/amplicon-nf_0.8.0 needed
ReferenceInsertion in PV750882.1 for sequence Noro-P20-30-new-184 NC_039475.1_artic-network/amplicon-nf_0.8.0 needed
ReferenceInsertion in PV784321.1 for sequence Noro-P79-30-new-186 NC_039475.1_artic-network/amplicon-nf_0.8.0 needed
ReferenceInsertion in PV750882.1 for sequence Noro-P38-30-new-184 NC_039475.1_artic-network/amplicon-nf_0.8.0 needed
ReferenceInsertion in PV784324.1 for sequence Noro-P104-30-new-

In [31]:
%%time
with open('g2_17_consensus_orf1_gb2seqed.fasta', 'wt') as fh:
    SeqIO.write(yield_orf_sequences('g2_17_consensus.fasta', "ORF1"), fh, 'fasta')

CPU times: user 98.7 ms, sys: 29.5 ms, total: 128 ms
Wall time: 5.01 s
